# Results: Maples KYC proof of concept

An investor uploads an ID and a subscription agreement. The pipeline

1. **classifies** each file (passport / ID card / subscription agreement) from its content,
2. **extracts** the KYC fields two independent ways per document,
3. **validates** each field (check digits, agreement between methods, ID vs subscription), and
4. gives every field a **traffic light**: green = confident, amber = analyst confirms, red = act on it.

Everything is scored against the answer key. All data is synthetic or public.

In [ ]:
from pathlib import Path
import os, sys
if Path(os.getcwd()).name == "notebooks":
    os.chdir("..")  # run from the repo root so the pipeline imports work
sys.path.insert(0, os.getcwd())
import pandas as pd
import matplotlib.pyplot as plt
import run_pipeline
from config import TODAY
from report import method_table

results, metrics, method_rows = run_pipeline.main()

## Evaluation

In [ ]:
pd.Series(metrics, name="result").to_frame()

## Method comparison

How often each method alone reads a field right. Subscription OCR is split by how the document arrived:
this is where messy real-world input shows up.

In [ ]:
table = method_table(method_rows)
summary = table.groupby(["document", "method", "condition"])[["correct", "total"]].sum()
summary["accuracy"] = summary["correct"] / summary["total"]
summary.style.format({"accuracy": "{:.0%}"})

In [ ]:
sub = summary.loc["subscription"]["accuracy"].unstack("method").reindex(["digital", "scanned", "photo"])
ax = sub.plot.bar(figsize=(6, 3.5), rot=0, color=["#1F3A5F", "#8FA9C8"])
ax.set_ylabel("fields read correctly")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_title("Subscription agreement: accuracy by method and input quality")
ax.legend(title=None, frameon=False)
plt.tight_layout()

## Investor tracker

In [ ]:
COLOURS = {"green": "#C6EFCE", "amber": "#FFEB9C", "red": "#FFC7CE"}
tracker = pd.DataFrame([{
    "investor": r["truth"]["investor_id"], "status": r["status"],
    "documents": ", ".join(f["predicted"] for f in r["files"]),
    "flags": sum(x["light"] != "green" for x in r["records"]),
    "answer key": r["truth"]["scenario"] + (" + expired ID" if r["truth"]["id"]["expiry_date"] < TODAY.isoformat() else ""),
} for r in results])
tracker.style.map(lambda v: f"background-color: {COLOURS.get(v, '')}", subset=["status"])

## Worked examples

In [ ]:
def field_view(investor_id):
    r = next(r for r in results if r["truth"]["investor_id"] == investor_id)
    df = pd.DataFrame(r["records"])[["document", "field", "value", "light", "note", "correct"]]
    return df.style.map(lambda v: f"background-color: {COLOURS.get(v, '')}", subset=["light"])

def first(scenario, condition=None):
    return next(r["truth"]["investor_id"] for r in results if r["truth"]["scenario"] == scenario
                and (condition is None or r["truth"]["condition"] == condition))

**A clean, digitally filled file.** Most fields green; anything amber says why.

In [ ]:
field_view(first("clean", "digital"))

**A date of birth that differs from the ID.**

In [ ]:
field_view(first("dob_mismatch"))

**A date of birth left blank.** The client's ask: flag what is missing.

In [ ]:
field_view(first("missing_dob"))